# Task 2: Baseline Models

Self-contained: no hidden src deps, no precomputed caches. Everything inline.

In [ ]:
import io, pickle, tarfile, urllib.request, urllib.parse
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from collections import Counter

ROOT = Path.cwd()
while not (ROOT / 'dataset-task2').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'dataset-task2'
CACHE = ROOT / '.cache'
CACHE.mkdir(exist_ok=True)

articles = pd.read_csv(DATA / 'articles.csv')
categories = pd.read_csv(DATA / 'categories.csv')
train = pd.read_csv(DATA / 'states_train.csv')
test = pd.read_csv(DATA / 'states_test.csv')
print(f'Loaded {len(articles)} articles, {len(train)} train, {len(test)} test states')

In [ ]:
def load_adjacency():
    for p in [CACHE / 'wikispeedia_adj.pkl', DATA / 'wikispeedia_adj.pkl']:
        if p.exists():
            with open(p, 'rb') as f:
                return pickle.load(f)
    title_to_id = dict(zip(articles['title'].str.strip(), articles['article_id']))
    url = 'https://snap.stanford.edu/data/wikispeedia/wikispeedia_paths-and-graph.tar.gz'
    print(f'Downloading Wikispeedia from {url} ...')
    with urllib.request.urlopen(url) as resp:
        with tarfile.open(fileobj=io.BytesIO(resp.read()), mode='r:gz') as tar:
            f = tar.extractfile('wikispeedia_paths-and-graph/links.tsv')
            content = f.read()
    links = pd.read_csv(io.BytesIO(content), sep='\t', skiprows=14,
                        header=None, names=['source', 'target'])
    links['sid'] = links['source'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links['tid'] = links['target'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links = links.dropna(subset=['sid', 'tid'])
    adj = {i: [] for i in range(len(articles))}
    for _, r in links.iterrows():
        adj[int(r['sid'])].append(int(r['tid']))
    adj = {k: list(set(v)) for k, v in adj.items()}
    with open(CACHE / 'wikispeedia_adj.pkl', 'wb') as f:
        pickle.dump(adj, f)
    return adj

adj = load_adjacency()
print(f'Adjacency: {len(adj)} nodes, {sum(len(v) for v in adj.values())} edges')

In [ ]:
if (CACHE / 'article_embeddings.npy').exists():
    embeddings = np.load(CACHE / 'article_embeddings.npy')
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(articles['title'].tolist(), show_progress_bar=True, batch_size=64)
    np.save(CACHE / 'article_embeddings.npy', embeddings)

all_cats = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_cats)}
cat_enc = np.zeros((len(articles), len(all_cats)), dtype=np.float32)
for _, r in categories.iterrows():
    cat_enc[r['article_id'], cat_to_idx[r['category']]] = 1.0
print(f'Embeddings: {embeddings.shape}, Categories: {cat_enc.shape}')

In [ ]:
def make_submission(state_ids, preds, path):
    pd.DataFrame({'state_id': state_ids, 'predicted_next_article_id': preds}).to_csv(path, index=False)

def validate(path):
    expected = pd.read_csv(DATA / 'sample_submission.csv')
    actual = pd.read_csv(path)
    assert list(actual.columns) == list(expected.columns)
    assert len(actual) == len(expected)
    assert list(actual['state_id']) == list(expected['state_id'])
    print(f'  Validated: {path}')

## Baseline 1: Title Similarity

In [ ]:
def predict_title_sim(df, adj, emb):
    preds = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc='Title sim'):
        links = adj.get(r['current_article_id'], [])
        if not links:
            preds.append(r['target_article_id']); continue
        scores = emb[links] @ emb[r['target_article_id']]
        preds.append(links[scores.argmax()])
    return np.array(preds)

tr = predict_title_sim(train, adj, embeddings)
print(f'Train acc: {(tr == train["next_article_id"].values).mean()*100:.2f}%')

te = predict_title_sim(test, adj, embeddings)
d = ROOT / 'submissions' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_title_sim'
d.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, te, d / 'submission.csv')
validate(d / 'submission.csv')

## Baseline 2: Category Overlap

In [ ]:
def predict_cat(df, adj, cat_enc):
    preds = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc='Category'):
        links = adj.get(r['current_article_id'], [])
        if not links:
            preds.append(r['target_article_id']); continue
        scores = (cat_enc[links] * cat_enc[r['target_article_id']][None, :]).sum(axis=1)
        preds.append(links[scores.argmax()])
    return np.array(preds)

tr = predict_cat(train, adj, cat_enc)
print(f'Train acc: {(tr == train["next_article_id"].values).mean()*100:.2f}%')

te = predict_cat(test, adj, cat_enc)
d = ROOT / 'submissions' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_category'
d.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, te, d / 'submission.csv')
validate(d / 'submission.csv')

## Baseline 3: Most Popular Link

In [ ]:
popularity = {}
for _, r in train.iterrows():
    popularity.setdefault(r['current_article_id'], Counter())[r['next_article_id']] += 1

def predict_popular(df, adj, pop):
    preds = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc='Popular'):
        c = r['current_article_id']
        if c in pop:
            preds.append(pop[c].most_common(1)[0][0])
        else:
            links = adj.get(c, [])
            preds.append(links[0] if links else r['target_article_id'])
    return np.array(preds)

tr = predict_popular(train, adj, popularity)
print(f'Train acc: {(tr == train["next_article_id"].values).mean()*100:.2f}%')

te = predict_popular(test, adj, popularity)
d = ROOT / 'submissions' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_popular'
d.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, te, d / 'submission.csv')
validate(d / 'submission.csv')

## Baseline 4: XGBoost

Generate features inline, no precomputed cache dependency.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Build candidate-level features inline
X_rows, y_rows, info = [], [], []
for _, r in tqdm(train.iterrows(), total=len(train), desc='XGB features'):
    curr, tgt, nxt = r['current_article_id'], r['target_article_id'], r['next_article_id']
    links = adj.get(curr, [])
    if not links:
        continue
    ce, te = embeddings[curr], embeddings[tgt]
    cc, tc = cat_enc[curr], cat_enc[tgt]
    for link in links:
        le, lc = embeddings[link], cat_enc[link]
        X_rows.append([
            float(ce @ te), float(le @ te), float(le @ ce),
            float((cc * tc).sum()), float((lc * tc).sum()),
            len(adj.get(link, [])), len(links),
            int(link == tgt), int(link == curr),
            float(ce @ le - ce @ te),
        ])
        y_rows.append(int(link == nxt))
        info.append(r['state_id'])

X = np.array(X_rows, dtype=np.float32)
y = np.array(y_rows, dtype=np.float32)
print(f'{len(X)} candidate-samples, {y.mean()*100:.2f}% positive')

In [ ]:
st = list(set(info))
tr_ids, vl_ids = train_test_split(st, test_size=0.2, random_state=42)
tr_ids = set(tr_ids)

tr_mask = np.array([i in tr_ids for i in info])
vl_mask = ~tr_mask

X_tr, y_tr = X[tr_mask], y[tr_mask]
X_vl, y_vl = X[vl_mask], y[vl_mask]

spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
print(f'Train: {len(X_tr)}, Val: {len(X_vl)}, pos_weight: {spw:.1f}')

xgbm = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=spw, eval_metric='logloss', random_state=42, n_jobs=-1)
xgbm.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=50)

In [ ]:
def predict_xgb(df, adj, emb, cat, model):
    preds = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc='XGBoost'):
        curr, tgt = r['current_article_id'], r['target_article_id']
        links = adj.get(curr, [])
        if not links:
            preds.append(tgt); continue
        ce, te = emb[curr], emb[tgt]
        cc, tc = cat[curr], cat[tgt]
        feats = []
        for link in links:
            le, lc = emb[link], cat[link]
            feats.append([
                float(ce @ te), float(le @ te), float(le @ ce),
                float((cc * tc).sum()), float((lc * tc).sum()),
                len(adj.get(link, [])), len(links),
                int(link == tgt), int(link == curr),
                float(ce @ le - ce @ te),
            ])
        scores = model.predict_proba(np.array(feats, dtype=np.float32))[:, 1]
        preds.append(links[scores.argmax()])
    return np.array(preds)

tr = predict_xgb(train, adj, embeddings, cat_enc, xgbm)
print(f'Train acc: {(tr == train["next_article_id"].values).mean()*100:.2f}%')

te = predict_xgb(test, adj, embeddings, cat_enc, xgbm)
d = ROOT / 'submissions' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_xgb'
d.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, te, d / 'submission.csv')
xgbm.save_model(str(d / 'xgb_model.json'))
validate(d / 'submission.csv')

## Baseline 5: Ensemble (majority vote)

In [ ]:
def predict_ensemble(df, p1, p2, p3):
    preds = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc='Ensemble'):
        v = Counter([p1(r), p2(r), p3(r)])
        preds.append(v.most_common(1)[0][0])
    return np.array(preds)

def _ptitle(r):
    links = adj.get(r['current_article_id'], [])
    if not links: return r['target_article_id']
    return links[(embeddings[links] @ embeddings[r['target_article_id']]).argmax()]

def _pcat(r):
    links = adj.get(r['current_article_id'], [])
    if not links: return r['target_article_id']
    s = (cat_enc[links] * cat_enc[r['target_article_id']][None, :]).sum(axis=1)
    return links[s.argmax()]

def _ppop(r):
    c = r['current_article_id']
    if c in popularity: return popularity[c].most_common(1)[0][0]
    links = adj.get(c, [])
    return links[0] if links else r['target_article_id']

tr = predict_ensemble(train, _ptitle, _pcat, _ppop)
print(f'Train acc: {(tr == train["next_article_id"].values).mean()*100:.2f}%')

te = predict_ensemble(test, _ptitle, _pcat, _ppop)
d = ROOT / 'submissions' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_ensemble'
d.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, te, d / 'submission.csv')
validate(d / 'submission.csv')

print('\nAll baselines complete.')